In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

mariamhany44_100_enhanced_final_path = kagglehub.dataset_download('mariamhany44/100-enhanced-final')

print('Data source import complete.')


Data source import complete.


In [5]:
# ============================================================
# TCN TRAINING USING DATA_DIR + YOUR EXACT MODEL
# ============================================================

import os
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# CONFIG & PATHS
# ============================================================

DATA_DIR = os.path.join(mariamhany44_100_enhanced_final_path, "balanced_dataset")
OUTPUT_DIR = "/content/training_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ============================================================
# LOAD DATA
# ============================================================

def load_split(split):
    X = np.load(os.path.join(DATA_DIR, split, "X.npy"))
    y = np.load(os.path.join(DATA_DIR, split, "y.npy"))
    m = np.load(os.path.join(DATA_DIR, split, "mask.npy"))
    return X, y, m

X_train, y_train, m_train = load_split("train")
X_val, y_val, m_val = load_split("val")
X_test, y_test, m_test = load_split("test")

FEATURE_DIM = X_train.shape[2]
TARGET_FRAMES = X_train.shape[1]
num_classes = len(np.unique(y_train))

print(f"\n📊 Data Info:")
print(f"   Feature dim: {FEATURE_DIM}")
print(f"   Frames: {TARGET_FRAMES}")
print(f"   Classes: {num_classes}")

# ============================================================
# DATASET
# ============================================================

class ASLDataset(Dataset):
    def __init__(self, data, masks, labels, augment=False):
        self.data = torch.from_numpy(data).float()
        self.masks = torch.from_numpy(masks).float()
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.data[idx]
        m = self.masks[idx]
        y = self.labels[idx]

        if self.augment:
            if torch.rand(1) > 0.5:
                x += torch.randn_like(x) * 0.01
            if torch.rand(1) > 0.5:
                x *= (torch.rand(1) * 0.2 + 0.9)

        return x.transpose(0, 1), m, y


BATCH_SIZE = min(8, len(X_train))

train_loader = DataLoader(ASLDataset(X_train, m_train, y_train, True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ASLDataset(X_val, m_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(ASLDataset(X_test, m_test, y_test), batch_size=BATCH_SIZE)

# ============================================================
# MODEL (EXACT SAME)
# ============================================================

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.Dropout(0.3),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.Dropout(0.3),
            nn.ReLU(inplace=True),
        )
        self.res = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        y = self.net(x)
        if y.size(2) != x.size(2):
            y = y[..., :x.size(2)]
        return y + self.res(x)


class TCN(nn.Module):
    def __init__(self, input_dim, num_classes, num_channels=64, num_layers=3):
        super().__init__()

        actual_layers = min(num_layers, len(X_train) // 10)
        channels = [num_channels] * max(1, actual_layers)

        layers = []
        for i, c in enumerate(channels):
            in_dim = input_dim if i == 0 else channels[i-1]
            layers.append(TemporalBlock(in_dim, c, dilation=2**i))

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], num_classes)
        self.dropout = nn.Dropout(0.4)

    def masked_pool(self, x, mask):
        mask = mask.unsqueeze(1)
        x = x * mask
        return x.sum(dim=2) / mask.sum(dim=2).clamp(min=1)

    def forward(self, x, mask):
        x = self.tcn(x)
        x = self.masked_pool(x, mask)
        x = self.dropout(x)
        return self.fc(x)


# dynamic sizing
if len(X_train) < 50:
    num_channels, num_layers = 32, 2
elif len(X_train) < 200:
    num_channels, num_layers = 64, 3
else:
    num_channels, num_layers = 128, 4

model = TCN(FEATURE_DIM, num_classes, num_channels, num_layers).to(DEVICE)

# ============================================================
# LOSS + OPTIMIZER (EXACT)
# ============================================================

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

class SmoothCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, logits, targets):
        n_classes = logits.size(1)

        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        log_probs = torch.log_softmax(logits, dim=1)

        if self.weight is not None:
            weight = self.weight[targets]
            loss = -(true_dist * log_probs).sum(dim=1) * weight
        else:
            loss = -(true_dist * log_probs).sum(dim=1)

        return loss.mean()


criterion = SmoothCrossEntropy(0.1, class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# ============================================================
# TRAINING LOOP (UNCHANGED)
# ============================================================

EPOCHS = 100
PATIENCE = 15
GRAD_CLIP = 1.0

best_val_acc = 0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

def train_one_epoch():
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x, m, y in train_loader:
        x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x, m)
        loss = criterion(out, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for x, m, y in loader:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            out = model(x, m)
            loss = criterion(out, y)

            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


print("\n🚀 START TRAINING")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)

    scheduler.step(val_loss)

    print(f"E{epoch+1:03} | Train {train_acc:.4f} | Val {val_acc:.4f}")

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, os.path.join(OUTPUT_DIR, "best_tcn_model.pth"))

        print("⭐ Best saved")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("⏹ Early stopping")
            break

# ============================================================
# TEST
# ============================================================

checkpoint = torch.load(os.path.join(OUTPUT_DIR, "best_tcn_model.pth"))
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc = evaluate(test_loader)

print(f"\n🎯 TEST ACC: {test_acc:.4f}")

# ============================================================
# SAVE
# ============================================================

np.save(os.path.join(OUTPUT_DIR, "label_encoder.npy"), np.unique(y_train))

with open(os.path.join(OUTPUT_DIR, "history.pkl"), "wb") as f:
    pickle.dump({
        "train_acc": train_accs,
        "val_acc": val_accs,
        "test_acc": test_acc
    }, f)

print(f"\n📁 Saved to {OUTPUT_DIR}")

Using device: cuda

📊 Data Info:
   Feature dim: 1482
   Frames: 125
   Classes: 105

🚀 START TRAINING
E001 | Train 0.0724 | Val 0.1956
⭐ Best saved
E002 | Train 0.2433 | Val 0.4089
⭐ Best saved
E003 | Train 0.4321 | Val 0.5756
⭐ Best saved
E004 | Train 0.5665 | Val 0.6489
⭐ Best saved
E005 | Train 0.6707 | Val 0.7200
⭐ Best saved
E006 | Train 0.7407 | Val 0.7089
E007 | Train 0.7899 | Val 0.7689
⭐ Best saved
E008 | Train 0.8185 | Val 0.7689
E009 | Train 0.8547 | Val 0.7978
⭐ Best saved
E010 | Train 0.8765 | Val 0.7911
E011 | Train 0.8955 | Val 0.8000
⭐ Best saved
E012 | Train 0.9048 | Val 0.7867
E013 | Train 0.9216 | Val 0.8200
⭐ Best saved
E014 | Train 0.9352 | Val 0.8400
⭐ Best saved
E015 | Train 0.9396 | Val 0.8756
⭐ Best saved
E016 | Train 0.9407 | Val 0.8622
E017 | Train 0.9491 | Val 0.8244
E018 | Train 0.9565 | Val 0.8733
E019 | Train 0.9578 | Val 0.8667
E020 | Train 0.9635 | Val 0.8578
E021 | Train 0.9597 | Val 0.8556
E022 | Train 0.9622 | Val 0.8889
⭐ Best saved
E023 | Train 0.